# 🟢 **Generate Test**

## 🔴 **Imports**

In [15]:
import os
import math
import re
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F
from tokenizers import Tokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 🔴 **CONFIG**

In [ ]:
MODEL_PATH = os.path.join(
    "/content/drive/MyDrive/ark-alpha-mini-v1-2.9M.pt",
)

TOKENIZER_PATH = os.path.join(
    "/content/drive/MyDrive/bpe-tokenizer_alphabet_nlp_dataset_10,000.json",
)


TEST_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

## 🔴 **START MODEL TEST**

In [21]:
print("=" * 100)
print("🧪 INDEPENDENT MODEL TEST")
print("=" * 100)

print(f"Device: {TEST_DEVICE}")
print(f"Tokenizer: {TOKENIZER_PATH}")
print(f"Model: {MODEL_PATH}")

🧪 INDEPENDENT MODEL TEST
Device: cpu
Tokenizer: tokenizer\bpe-tokenizer_alphabet_nlp_dataset_10,000.json
Model: ark-alpha-mini-v1-2.9M.pt


## 🔴 **FILE CHECK**

In [23]:
if not os.path.exists(TOKENIZER_PATH):
    raise FileNotFoundError(
        f"❌ Tokenizer not found:\n{TOKENIZER_PATH}"
    )

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"❌ Model not found:\n{MODEL_PATH}"
    )

print("✅ Tokenizer file found.")
print("✅ Model file found.")

✅ Tokenizer file found.
✅ Model file found.


## 🔴 **LOAD TOKENIZER**

In [25]:
test_tokenizer = Tokenizer.from_file(
    TOKENIZER_PATH
)

print("✅ Tokenizer loaded successfully.")
print(
    f"Vocabulary size: "
    f"{test_tokenizer.get_vocab_size():,}"
)

✅ Tokenizer loaded successfully.
Vocabulary size: 10,000


## 🔴 **MULTI-HEAD ATTENTION**

In [26]:
class IndependentMultiHeadAttention(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head,
        dropout_rate
    ):
        super().__init__()

        if n_embd % n_head != 0:
            raise ValueError(
                "n_embd must be divisible by n_head."
            )

        self.n_embd = n_embd
        self.n_head = n_head
        self.head_size = n_embd // n_head

        self.qkv_proj = nn.Linear(
            n_embd,
            3 * n_embd,
            bias=False
        )

        self.c_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.c_proj.residual = True

    def forward(self, x):

        B, T, C = x.shape

        qkv = (
            self.qkv_proj(x)
            .view(
                B,
                T,
                3,
                self.n_head,
                self.head_size
            )
            .permute(
                2,
                0,
                3,
                1,
                4
            )
        )

        q, k, v = qkv.unbind(0)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True
        )

        y = (
            y.transpose(1, 2)
            .contiguous()
            .view(
                B,
                T,
                C
            )
        )

        return self.c_proj(y)

## 🔴 **FEED FORWARD**

In [27]:
class IndependentFeedForward(nn.Module):

    def __init__(
        self,
        n_embd,
        f_expnd,
        dropout_rate
    ):
        super().__init__()

        hidden_size = int(
            f_expnd * n_embd
        )

        self.up_proj = nn.Linear(
            n_embd,
            hidden_size,
            bias=False
        )

        self.down_proj = nn.Linear(
            hidden_size,
            n_embd,
            bias=False
        )

        self.down_proj.residual = True

        self.mlp_dropout = nn.Dropout(
            dropout_rate
        )

    def forward(self, x):

        return self.mlp_dropout(
            self.down_proj(
                F.gelu(
                    self.up_proj(x)
                )
            )
        )

## 🔴 **DECODER BLOCK**

In [28]:
class IndependentDecoderBlock(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head,
        f_expnd,
        dropout_rate
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            n_embd
        )

        self.mha = IndependentMultiHeadAttention(
            n_embd,
            n_head,
            dropout_rate
        )

        self.ln2 = nn.LayerNorm(
            n_embd
        )

        self.mlp = IndependentFeedForward(
            n_embd,
            f_expnd,
            dropout_rate
        )

        self.dropout = nn.Dropout(
            dropout_rate
        )

    def forward(self, x):

        x = x + self.dropout(
            self.mha(
                self.ln1(x)
            )
        )

        x = x + self.dropout(
            self.mlp(
                self.ln2(x)
            )
        )

        return x

## 🔴 **ARK MODEL**

In [29]:
class IndependentARK(nn.Module):

    def __init__(
        self,
        vocab_size=10000,
        max_seq_len=1024,
        n_layer=8,
        n_head=16,
        n_embd=128,
        f_expnd=4,
        dropout_rate=0.2
    ):
        super().__init__()

        self.wte = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.wpe = nn.Embedding(
            max_seq_len,
            n_embd
        )

        self.decoders = nn.ModuleList(
            [
                IndependentDecoderBlock(
                    n_embd=n_embd,
                    n_head=n_head,
                    f_expnd=f_expnd,
                    dropout_rate=dropout_rate
                )
                for _ in range(n_layer)
            ]
        )

        self.lnf = nn.LayerNorm(
            n_embd
        )

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = self.wte.weight

    def forward(self, idx):

        B, T = idx.shape

        if T > self.wpe.num_embeddings:
            raise ValueError(
                f"Sequence length {T} exceeds "
                f"max_seq_len={self.wpe.num_embeddings}"
            )

        positions = torch.arange(
            T,
            device=idx.device
        )

        x = (
            self.wte(idx)
            +
            self.wpe(positions)
        )

        for decoder in self.decoders:
            x = decoder(x)

        x = self.lnf(x)

        return self.lm_head(x)

## 🔴 **CREATE MODEL**

In [34]:
test_model = IndependentARK(
    vocab_size=10_000,
    max_seq_len=128,
    n_layer=8,
    n_head=16,
    n_embd=128,
    f_expnd=4,
    dropout_rate=0.2
).to(TEST_DEVICE)

## 🔴 **PARAMETER COUNT**

In [35]:
total_params = sum(
    p.numel()
    for p in test_model.parameters()
)

print(
    f"📊 Parameters: "
    f"{total_params:,} "
    f"({total_params / 1e6:.2f}M)"
)

📊 Parameters: 2,873,600 (2.87M)


## 🔴 **LOAD CHECKPOINT**

In [40]:
print(
    "🔄 Loading model.pt..."
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=TEST_DEVICE
)

if not isinstance(checkpoint, dict):
    raise TypeError(
        "❌ Unexpected checkpoint format."
    )

print("\n🔍 Checkpoint information")

print("Checkpoint keys:")
for key in checkpoint.keys():
    print(f"   • {key}")

# Display training progress information
if "optimizer_step" in checkpoint:
    print(
        f"\n📈 Optimizer step: "
        f"{checkpoint['optimizer_step']:,}"
    )
else:
    print(
        "\n⚠️ 'optimizer_step' not found in checkpoint."
    )

if "seen_tokens" in checkpoint:
    print(
        f"📊 Seen tokens: "
        f"{checkpoint['seen_tokens']:,}"
    )
else:
    print(
        "⚠️ 'seen_tokens' not found in checkpoint."
    )

if "model_state_dict" not in checkpoint:
    raise KeyError(
        "❌ 'model_state_dict' not found in checkpoint."
    )

test_model.load_state_dict(
    checkpoint["model_state_dict"]
)

test_model.eval()

print(
    "\n✅ model.pt loaded successfully."
)

🔄 Loading model.pt...

🔍 Checkpoint information
Checkpoint keys:
   • model_state_dict
   • optimizer_state_dict
   • optimizer_step
   • seen_tokens

📈 Optimizer step: 3,337
📊 Seen tokens: 1,708,288

✅ model.pt loaded successfully.


## 🔴 **CHECKPOINT VERIFICATION**

In [42]:
checkpoint_state = checkpoint[
    "model_state_dict"
]

model_state = test_model.state_dict()

checkpoint_keys = set(
    checkpoint_state.keys()
)

model_keys = set(
    model_state.keys()
)

missing_keys = (
    model_keys -
    checkpoint_keys
)

unexpected_keys = (
    checkpoint_keys -
    model_keys
)

shape_mismatches = []

for key in checkpoint_keys & model_keys:

    if (
        checkpoint_state[key].shape
        !=
        model_state[key].shape
    ):
        shape_mismatches.append(
            (
                key,
                checkpoint_state[key].shape,
                model_state[key].shape
            )
        )

print(
    "🔍 Checkpoint verification"
)

print(
    f"Checkpoint tensors: "
    f"{len(checkpoint_keys):,}"
)

print(
    f"Model tensors:      "
    f"{len(model_keys):,}"
)

if missing_keys:
    print("\n❌ Missing keys:")
    for key in sorted(missing_keys):
        print("   ", key)
else:
    print("✅ No missing keys.")

if unexpected_keys:
    print("\n⚠️ Unexpected keys:")
    for key in sorted(unexpected_keys):
        print("   ", key)
else:
    print("✅ No unexpected keys.")

if shape_mismatches:
    print("\n❌ Shape mismatches:")
    for (
        key,
        checkpoint_shape,
        model_shape
    ) in shape_mismatches:
        print(
            f"   {key}: "
            f"checkpoint={checkpoint_shape}, "
            f"model={model_shape}"
        )
else:
    print("✅ All tensor shapes match.")

if (
    not missing_keys
    and not unexpected_keys
    and not shape_mismatches
):
    print(
        "\n🎯 Checkpoint architecture "
        "matches the model perfectly."
    )
else:
    raise RuntimeError(
        "\n❌ Checkpoint is not fully compatible."
    )

🔍 Checkpoint verification
Checkpoint tensors: 69
Model tensors:      69
✅ No missing keys.
✅ No unexpected keys.
✅ All tensor shapes match.

🎯 Checkpoint architecture matches the model perfectly.


## 🔴 **GENERATION FUNCTION**

In [43]:
def independent_generate(
    model,
    tokenizer,
    prompt,
    max_seq_len=128,
    temperature=0.7,
    top_k=10,
    greedy=False,
    device="cuda",
    seed=42
):

    model.eval()

    prompt_ids = tokenizer.encode(
        prompt
    ).ids

    if len(prompt_ids) == 0:
        raise ValueError(
            "❌ Prompt produced zero tokens."
        )

    if len(prompt_ids) >= max_seq_len:
        raise ValueError(
            "❌ Prompt is already equal to "
            "or longer than max_seq_len."
        )

    inputs = torch.tensor(
        prompt_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    rng = torch.Generator(
        device=device
    )

    rng.manual_seed(seed)

    with torch.no_grad():

        while inputs.shape[1] < max_seq_len:

            logits = model(
                inputs
            )

            next_logits = logits[
                :,
                -1,
                :
            ]

            if greedy:

                next_token = torch.argmax(
                    next_logits,
                    dim=-1
                )

            else:

                temperature = max(
                    temperature,
                    1e-5
                )

                probs = torch.softmax(
                    next_logits / temperature,
                    dim=-1
                )

                k = min(
                    top_k,
                    probs.shape[-1]
                )

                topk_probs, topk_indices = (
                    torch.topk(
                        probs,
                        k=k,
                        dim=-1
                    )
                )

                # Important:
                # torch.multinomial expects a valid probability
                # distribution. Re-normalize top-k probabilities.
                topk_probs = (
                    topk_probs
                    /
                    topk_probs.sum(
                        dim=-1,
                        keepdim=True
                    )
                )

                sampled = torch.multinomial(
                    topk_probs,
                    num_samples=1,
                    generator=rng
                )

                next_token = torch.gather(
                    topk_indices,
                    -1,
                    sampled
                ).squeeze(-1)

            inputs = torch.cat(
                [
                    inputs,
                    next_token.unsqueeze(-1)
                ],
                dim=-1
            )

    generated_ids = inputs[
        0,
        len(prompt_ids):
    ].tolist()

    return tokenizer.decode(
        generated_ids
    )

## 🔴 **TEST PROMPT**

In [44]:
TEST_PROMPT = (
    "یادگیری ماشین (Machine Learning - ML)، "
    "یکی از زیرشاخه‌های هوش مصنوعی است که"
)

## 🔴 **NEXT TOKEN PROBABILITY TEST**

In [45]:
print("=" * 100)
print("🔬 NEXT TOKEN PROBABILITY TEST")
print("=" * 100)

prompt = (
    "یادگیری ماشین (Machine Learning - ML)، "
    "یکی از زیرشاخه‌های هوش مصنوعی است که"
)

print("\n[Prompt]")
print(prompt)

ids = test_tokenizer.encode(
    prompt
).ids

print("\n[Prompt Token IDs]")
print(ids)

print(
    f"\nPrompt token count: "
    f"{len(ids)}"
)

x = torch.tensor(
    [ids],
    dtype=torch.long,
    device=TEST_DEVICE
)

with torch.no_grad():
    logits = test_model(x)

next_logits = logits[
    :,
    -1,
    :
]

probs = torch.softmax(
    next_logits,
    dim=-1
)

top_probs, top_ids = torch.topk(
    probs,
    k=min(20, probs.shape[-1]),
    dim=-1
)

print("\n")
print("-" * 100)
print("TOP-20 NEXT TOKEN PREDICTIONS")
print("-" * 100)
print(
    f"{'Rank':>4} "
    f"{'ID':>6} "
    f"{'Probability':>14} "
    f"{'Token':>30}"
)
print("-" * 100)

for rank, (p, idx) in enumerate(
    zip(
        top_probs[0],
        top_ids[0]
    ),
    start=1
):
    token = test_tokenizer.decode(
        [idx.item()]
    )

    print(
        f"{rank:4d} "
        f"{idx.item():6d} "
        f"{p.item():14.8f} "
        f"{repr(token):>30}"
    )

🔬 NEXT TOKEN PROBABILITY TEST

[Prompt]
یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه‌های هوش مصنوعی است که

[Prompt Token IDs]
[681, 806, 265, 5041, 3120, 362, 792, 45, 614, 652, 222, 490, 9557, 1409, 151, 219, 464, 534, 221, 237]

Prompt token count: 20


----------------------------------------------------------------------------------------------------
TOP-20 NEXT TOKEN PREDICTIONS
----------------------------------------------------------------------------------------------------
Rank     ID    Probability                          Token
----------------------------------------------------------------------------------------------------
   1    212     0.06069844                          ' به'
   2    213     0.04946731                          ' در'
   3    222     0.03962061                          ' از'
   4    244     0.03290967                         ' این'
   5    273     0.01497572                          ' یک'
   6    210     0.01385280                          '

## 🔴 **EXPECTED CONTINUATION TEST**

In [46]:
print("=" * 100)
print("🎯 EXPECTED NEXT TOKEN TEST")
print("=" * 100)

expected = (
    " به سیستم‌ها اجازه می‌دهد"
)

expected_ids = test_tokenizer.encode(
    expected
).ids

print("\n[Expected continuation]")
print(repr(expected))

print("\n[Expected token IDs]")
print(expected_ids)

print("\n[Expected tokens]")

for position, idx in enumerate(
    expected_ids,
    start=1
):
    token = test_tokenizer.decode(
        [idx]
    )

    print(
        f"{position:3d}. "
        f"ID={idx:5d} "
        f"Token={repr(token)}"
    )

if len(expected_ids) > 0:

    expected_next_id = expected_ids[0]

    expected_prob = (
        probs[
            0,
            expected_next_id
        ].item()
    )

    print("\n")
    print("-" * 100)
    print("ACTUAL EXPECTED NEXT TOKEN")
    print("-" * 100)

    print(
        "Expected token ID:",
        expected_next_id
    )

    print(
        "Expected token:",
        repr(
            test_tokenizer.decode(
                [expected_next_id]
            )
        )
    )

    print(
        "Probability:",
        f"{expected_prob:.10f}"
    )

    sorted_ids = torch.argsort(
        probs[0],
        descending=True
    )

    expected_rank_tensor = torch.where(
        sorted_ids == expected_next_id
    )[0]

    if len(expected_rank_tensor) > 0:
        expected_rank = (
            expected_rank_tensor[0].item()
            + 1
        )

        print(
            "Rank:",
            expected_rank
        )
    else:
        print("Rank: Not found")

🎯 EXPECTED NEXT TOKEN TEST

[Expected continuation]
' به سیستم\u200cها اجازه می\u200cدهد'

[Expected token IDs]
[212, 412, 1409, 151, 239, 1734, 210, 1409, 151, 511]

[Expected tokens]
  1. ID=  212 Token=' به'
  2. ID=  412 Token=' سیستم'
  3. ID= 1409 Token='�'
  4. ID=  151 Token='�'
  5. ID=  239 Token='ها'
  6. ID= 1734 Token=' اجازه'
  7. ID=  210 Token=' می'
  8. ID= 1409 Token='�'
  9. ID=  151 Token='�'
 10. ID=  511 Token='دهد'


----------------------------------------------------------------------------------------------------
ACTUAL EXPECTED NEXT TOKEN
----------------------------------------------------------------------------------------------------
Expected token ID: 212
Expected token: ' به'
Probability: 0.0606984422
Rank: 1


## 🔴 **GREEDY DECODING**

In [47]:
print("=" * 100)
print("🧠 GREEDY DECODING")
print("=" * 100)

greedy_output = independent_generate(
    model=test_model,
    tokenizer=test_tokenizer,
    prompt=TEST_PROMPT,
    max_seq_len=128,
    greedy=True,
    device=TEST_DEVICE
)

print("\n[Prompt]")
print(TEST_PROMPT)

print("\n[Generated continuation]")
print(greedy_output)

🧠 GREEDY DECODING

[Prompt]
یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه‌های هوش مصنوعی است که

[Generated continuation]
 به‌ی آن‌ی این‌ی این‌ی این‌ی این‌ی این‌ی‌ی «ی‌ی این این این‌ی این‌ی «































































## 🔴 **TOP-K SAMPLING**

In [48]:
print("=" * 100)
print("🎲 TOP-K SAMPLING")
print("=" * 100)

for sample_number in range(
    1,
    4
):

    output = independent_generate(
        model=test_model,
        tokenizer=test_tokenizer,
        prompt=TEST_PROMPT,
        max_seq_len=128,
        temperature=0.7,
        top_k=10,
        greedy=False,
        device=TEST_DEVICE,
        seed=42 + sample_number
    )

    print(
        f"\n[Sample {sample_number}]"
    )

    print(output)

🎲 TOP-K SAMPLING

[Sample 1]
 از می‌های به‌های این‌های آن، به‌ی «�ی آن‌ها، در آن است.

**




-‌های استفاده این‌ی آیه‌های آن‌های آن‌ها. « «های « این به‌های به‌های استفادهی الل است.
```




-‌یکند.

**


- این می‌ها.




[Sample 2]
 از این‌ی این‌ی آن‌ی‌ی آن‌ی این‌های برنامه‌ها، به‌ای، با این‌های زبان‌شود.
```
**


- **ها، با این‌های این به‌ها این‌های یک این‌دهد.


- یک این‌ی «ها و به‌ی‌های استفاده، یک می‌های برنامه‌ی:
در

[Sample 3]
 به‌های یک برنامه‌کند.


-






در‌ی برنامه‌ی.
- **ی این‌ها به‌های «های یک این‌های به‌ی�های «�ی.
-
- ** **مِ این‌ی آن‌های این‌های یک�کند.

```

**





- **�ی یک آن‌یهِلَ


## 🔴 **TOKENIZER TEST**

In [49]:
print("=" * 100)
print("✅ TOKENIZER TEST")
print("=" * 100)

texts = [
    "یادگیری ماشین",
    (
        "یادگیری ماشین "
        "(Machine Learning - ML)، "
        "یکی از زیرشاخه‌های هوش مصنوعی است که"
    )
]

for text in texts:

    enc = test_tokenizer.encode(
        text
    )

    print("\nTEXT:")
    print(text)

    print("IDS:")
    print(enc.ids)

    print("TOKENS:")
    print(enc.tokens)

✅ TOKENIZER TEST

TEXT:
یادگیری ماشین
IDS:
[681, 806]
TOKENS:
['ĠÛĮØ§Ø¯Ú¯ÛĮØ±ÛĮ', 'ĠÙħØ§Ø´ÛĮÙĨ']

TEXT:
یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه‌های هوش مصنوعی است که
IDS:
[681, 806, 265, 5041, 3120, 362, 792, 45, 614, 652, 222, 490, 9557, 1409, 151, 219, 464, 534, 221, 237]
TOKENS:
['ĠÛĮØ§Ø¯Ú¯ÛĮØ±ÛĮ', 'ĠÙħØ§Ø´ÛĮÙĨ', 'Ġ(', 'Machine', 'ĠLearning', 'Ġ-', 'ĠM', 'L', ')ØĮ', 'ĠÛĮÚ©ÛĮ', 'ĠØ§Ø²', 'ĠØ²ÛĮØ±', 'Ø´Ø§Ø®Ùĩ', 'âĢ', 'Į', 'ÙĩØ§ÛĮ', 'ĠÙĩÙĪØ´', 'ĠÙħØµÙĨÙĪØ¹ÛĮ', 'ĠØ§Ø³Øª', 'ĠÚ©Ùĩ']


## 🔴 **TOKENIZER ROUND-TRIP TEST**

In [50]:
print("=" * 80)
print("🔬 TOKENIZER ROUND-TRIP TEST")
print("=" * 80)

for text in texts:

    enc = test_tokenizer.encode(
        text
    )

    decoded = test_tokenizer.decode(
        enc.ids
    )

    print("\nOriginal:")
    print(repr(text))

    print("Decoded:")
    print(repr(decoded))

    print(
        "MATCH:",
        text == decoded
    )

🔬 TOKENIZER ROUND-TRIP TEST

Original:
'یادگیری ماشین'
Decoded:
' یادگیری ماشین'
MATCH: False

Original:
'یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه\u200cهای هوش مصنوعی است که'
Decoded:
' یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه\u200cهای هوش مصنوعی است که'
MATCH: False


## 🔴 **FULL TARGET TEST**

In [51]:
target = """یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه‌های هوش مصنوعی است که به سیستم‌ها اجازه می‌دهد بدون برنامه‌ریزی صریح، از داده‌ها یاد بگیرند و بهبود یابند.

الگوریتم‌های یادگیری ماشین به سه دسته‌ی اصلی تقسیم می‌شوند:

- **یادگیری نظارت‌شده (Supervised Learning)**
- **یادگیری بدون نظارت (Unsupervised Learning)**
- **یادگیری تقویتی (Reinforcement Learning)**"""

enc = test_tokenizer.encode(
    target
)

decoded = test_tokenizer.decode(
    enc.ids
)

print("=" * 100)
print("📚 FULL TARGET TOKENIZER TEST")
print("=" * 100)

print("\nOriginal:")
print(repr(target))

print("\nDecoded:")
print(repr(decoded))

print(
    "\nMATCH:",
    target == decoded
)

print(
    "Token count:",
    len(enc.ids)
)

📚 FULL TARGET TOKENIZER TEST

Original:
'یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه\u200cهای هوش مصنوعی است که به سیستم\u200cها اجازه می\u200cدهد بدون برنامه\u200cریزی صریح، از داده\u200cها یاد بگیرند و بهبود یابند.\n\nالگوریتم\u200cهای یادگیری ماشین به سه دسته\u200cی اصلی تقسیم می\u200cشوند:\n\n- **یادگیری نظارت\u200cشده (Supervised Learning)**\n- **یادگیری بدون نظارت (Unsupervised Learning)**\n- **یادگیری تقویتی (Reinforcement Learning)**'

Decoded:
' یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه\u200cهای هوش مصنوعی است که به سیستم\u200cها اجازه می\u200cدهد بدون برنامه\u200cریزی صریح، از داده\u200cها یاد بگیرند و بهبود یابند.\n\nالگوریتم\u200cهای یادگیری ماشین به سه دسته\u200cی اصلی تقسیم می\u200cشوند:\n\n- **یادگیری نظارت\u200cشده (Supervised Learning)**\n- **یادگیری بدون نظارت (Unsupervised Learning)**\n- **یادگیری تقویتی (Reinforcement Learning)**'

MATCH: False
Token count: 104
